# Handoff Callbacks - Lifecycle Hooks

## Purpose
Learn how to add custom logic before agent handoffs execute. This enables audit logging, data prefetching, metrics emission, and triggering side workflows when agents hand off control.

## Key Concepts
- **on_handoff**: Async callback function executed before handoff
- **handoff()**: Function to customize handoff behavior with callbacks
- **input_type**: Pydantic model for type-safe handoff data
- **Audit Logging**: Track which agents are handling requests

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `handoff` function and `RunContextWrapper` for lifecycle hooks:

In [ ]:
import asyncio

from agents import Agent, Runner, handoff, RunContextWrapper
from pydantic import BaseModel

## Step 1: Create Specialist Agents

Define specialist agents as usual with `handoff_description`:

In [ ]:
billing_agent = Agent(
    name="Billing agent",
    handoff_description="Handles invoices, charges, and billing questions.",
    instructions="You are the billing specialist. Answer billing and invoice questions.",
    model=model_id,
)

refund_agent = Agent(
    name="Refund agent",
    handoff_description="Handles refund requests and processes returns.",
    instructions="You are the refund specialist. Help the customer get a refund.",
    model=model_id,
)

## Step 2: Define Handoff Callback

Create a callback function that executes before the handoff. This is perfect for:
- **Audit logging**: Track which agents handle which requests
- **Data prefetching**: Load data the agent will need
- **Metrics emission**: Send telemetry to observability systems
- **Side workflows**: Trigger notifications or other processes

💡 **Important**: Define an `input_type` (Pydantic model) to get type-safe handoff data.

In [ ]:
class RefundReason(BaseModel):
    reason: str

async def on_refund_handoff(ctx: RunContextWrapper[None], data: RefundReason):
    """Called before handing off to refund agent.
    
    Use cases:
    - Audit Log: Track refund requests
    - Prefetch data: Load customer history
    - Emit metrics: Count refund requests
    - Trigger workflow: Notify finance team
    """
    print(f"[handoff] Routing to Refund agent. Reason: {data.reason}")

## Step 3: Create Triage Agent with Callbacks

Use the `handoff()` function to customize specific handoffs:
- Regular agents can be added directly (like `billing_agent`)
- Use `handoff()` wrapper for callbacks and custom behavior

🔍 **Watch**: The `on_handoff` callback will print before refund agent executes!

In [ ]:
triage_agent = Agent(
    name="Triage agent",
    instructions=f"You triage customer requests. Hand off to the billing agent for "
                 "billing/invoice questions, and to the refund agent for refunds.",
    handoffs=[
        billing_agent,                    # Simple handoff (no callback)
        handoff(                          # Customized handoff with callback
            agent=refund_agent,
            on_handoff=on_refund_handoff,
            input_type=RefundReason,
        ),
    ],
    model=model_id,
)

## Step 4: Run and Observe Callbacks

When the triage agent hands off to the refund agent, you'll see the callback execute first:

⚡ **Execution Flow**:
1. User request → Triage agent
2. Triage decides to hand off to refund agent
3. **`on_refund_handoff` callback executes** (audit log prints)
4. Refund agent processes request
5. Response returns to user

In [ ]:
result = await Runner.run(
    triage_agent,
    "I was charged twice for my subscription and I want my money back."
)
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Handoff Callbacks** notebook!

### What You Learned
- How to use the `handoff()` function to customize handoffs
- Creating `on_handoff` callbacks for lifecycle hooks
- Using `input_type` for type-safe handoff data
- Real-world use cases: audit logging, data prefetching, metrics, notifications
- Mixing simple and customized handoffs in one triage agent